In [1]:
import json, pandas as pd
from pathlib import Path
R=Path('results') if Path('results').exists() else Path('experiments/E13B_evidence_evaluator_hardening/results')
S=json.load(open(R/'rescoring_summary.json'))['units']; ST=json.load(open(R/'stability_checks.json')); MA=json.load(open(R/'manual_audit.json'))
recs=json.load(open(R/'rescoring_case_records.json')); pd.set_option('display.width',230,'display.max_colwidth',110)

# E13B — Evidence evaluator hardening (evidence_evaluator_v2)
**Zero model calls; no TEST access; stored TRAIN/DEV outputs and unit-test fixtures only.**
## 1. Research question
Can the evidence evaluator be made robust to harmless formatting differences without changing the semantic standard for supporting evidence?
## 2. Why (E13 artefact)
Exact-substring mapping failed on correct quotes whose whitespace/line breaks differed (E13: 5 FULL, 1 RAG cases).
## 3. Frozen v2 specification
Exact-first, then formatting-normalized fallback (NFC; delete U+200B/C/D/2060/FEFF; collapse `str.isspace()` runs to one space; trim) with a normalized→original offset map; v1 first-occurrence-per-text policy preserved; τ=0.5 and joint semantics unchanged. No case-folding, punctuation edits, ellipsis segmentation, fuzzy or semantic matching.

In [2]:
print(open('../../evaluation/evidence_matching.py').read()[:3200]) if Path('../../evaluation/evidence_matching.py').exists() else print(open('evaluation/evidence_matching.py').read()[:3200])

"""
Canonical evidence->source-span matching (evidence_evaluator_v1 / v2). Deterministic, no model calls.

v1  (historical): exact verbatim substring match; kept as ``evidence_to_span_indices_v1`` so old results stay reproducible.
v2  (E13B):       exact-first, then a FORMATTING-normalized fallback whose match is mapped back to ORIGINAL character offsets, so the unchanged
                  gold-span overlap semantics keep operating in original-document coordinates.

Normalization (spec frozen in experiments/E13B_evidence_evaluator_hardening/config.yaml BEFORE historical re-scoring; applied identically to quote and context):
  A. Unicode NFC per base+combining-mark cluster (NOT NFKC: compatibility mappings could change numbers/semantics).
  B. delete zero-width formatting chars U+200B, U+200C, U+200D, U+2060, U+FEFF.
  C. every ``str.isspace()`` char (space, tab, CR/LF, NBSP, U+2000-200A, U+2028/9, U+202F, U+205F, U+3000, ...) -> runs collapse to ONE space.
  D. trim.
  NOT done: case-f

## 4. Unit tests
40 evaluator tests (`tests/test_evidence_matching.py`): exact/whitespace/zero-width/Unicode/offset/multi-occurrence/joint-semantics/v1-compat plus 9 adversarial negatives (negation, number, party, legal term, paraphrase, case, punctuation, missing word) — all pass; full suite 378/378.
## 5. Historical re-scoring v1 → v2 (stored outputs; v1 reproduces every reported joint)

In [3]:
pd.DataFrame([{ 'exp':u['exp'],'arm':u['arm'],'v1==reported':u['v1_matches_reported'],'joint v1':u['joint_v1'],'joint v2':u['joint_v2'],'recall v1':round(u['evidence_recall_v1'],3),'recall v2':round(u['evidence_recall_v2'],3),'prec v1':round(u['evidence_precision_v1'],3),'prec v2':round(u['evidence_precision_v2'],3),'src-valid v1':u['source_valid_records_v1'],'src-valid v2':u['source_valid_records_v2'],'mappings changed':len(u['mappings_changed']),'regressions':len(u['regress_span_not_superset'])+len(u['regress_source_valid_true_to_false'])+len(u['joint_success_to_fail'])} for u in S])

,exp,arm,v1==reported,joint v1,joint v2,recall v1,recall v2,prec v1,prec v2,src-valid v1,src-valid v2,mappings changed,regressions
0,E05,qwen_full_context,True,42,43,0.25,0.26,0.287,0.299,53,57,4,0
1,E07,qwen_rag_top5,True,50,51,0.29,0.31,0.358,0.383,49,54,5,0
2,E08B,gpt5mini_rag_top5,True,111,113,0.86,0.88,0.835,0.854,97,100,3,0
3,E11,A2_control,True,111,113,0.86,0.88,0.835,0.854,97,100,3,0
4,E11,A3_selective_agent,True,111,113,0.84,0.86,0.824,0.843,93,97,4,0
5,E12A,gpt5mini_top11,True,117,119,0.91,0.93,0.843,0.861,102,106,4,0
6,E12B,gpt_p0,True,114,116,0.85,0.87,0.825,0.845,100,102,2,0
7,E12B,gpt_p1,True,112,113,0.85,0.86,0.802,0.811,103,104,1,0
8,E12B,gpt_p2,True,106,107,0.85,0.86,0.787,0.796,104,105,1,0
9,E12B,gpt_p3,True,122,122,0.92,0.92,0.885,0.885,103,103,0,0


## 6. Manual audit of every changed case

In [4]:
print(json.dumps(MA,indent=1))

{
 "reviewed": "every changed record (38 unique; 41 unit-level rescued quotes; E11-A2 duplicates E08B's 3 records) inspected: model quote vs recovered original NDA text vs normalization responsible vs recovered span vs gold overlap",
 "classification": {
  "TRUE_FORMATTING_RESCUE": 38,
  "FALSE_POSITIVE": 0,
  "AMBIGUOUS_multi_occurrence": 0
 },
 "transformations_responsible": "line-break/whitespace-run merges in 38/38 (one, E13 69::nda-2, also contains zero-width spaces); no NFC or NBSP-only case occurred in the historical data",
 "automatic_check": "for every rescued quote, quote and recovered original slice are identical after deleting all whitespace/zero-width chars and NFC (formatting_only_verified=True for all 41); n_occurrences==1 for all 41",
 "observations": [
  "Many rescued quotes are long merged multi-sentence quotes that map onto several annotated spans (e.g. E13 FULL 13::nda-12 -> spans 93-102). That is v1's existing overlap semantics for any verbatim multi-span quote, un

In [5]:
rows=[]
for k,rs in recs.items():
    if k.startswith('E11::A2'): continue
    for r in rs:
        if r['v1_spans']!=r['v2_spans']: rows.append({'run':k,'case':r['case_id'],'gold':r['gold_label'],'pred':r['predicted_label'],'joint v1->v2':f"{r['joint_v1']}->{r['joint_v2']}",'causes':','.join(r['rescued_quotes'][0]['causes']) if r['rescued_quotes'] else '','formatting_only':all(d['formatting_only_verified'] for d in r['rescued_quotes']),'occ':max(d['n_occurrences'] for d in r['rescued_quotes'])})
pd.DataFrame(rows)

,run,case,gold,pred,joint v1->v2,causes,formatting_only,occ
0,E05::qwen_full_context,train::194::nda-10,Entailment,Entailment,False->False,"line_break_or_tab,whitespace_run_or_join",True,1
1,E05::qwen_full_context,train::352::nda-12,Entailment,Entailment,False->True,"line_break_or_tab,whitespace_run_or_join",True,1
2,E05::qwen_full_context,train::94::nda-16,NotMentioned,Entailment,False->False,"line_break_or_tab,whitespace_run_or_join",True,1
3,E05::qwen_full_context,train::350::nda-10,NotMentioned,Contradiction,False->False,"line_break_or_tab,whitespace_run_or_join",True,1
4,E07::qwen_rag_top5,train::161::nda-7,Contradiction,Contradiction,False->True,"line_break_or_tab,whitespace_run_or_join",True,1
5,E07::qwen_rag_top5,train::317::nda-13,Entailment,Contradiction,False->False,"line_break_or_tab,whitespace_run_or_join",True,1
6,E07::qwen_rag_top5,train::619::nda-10,Entailment,Contradiction,False->False,"line_break_or_tab,whitespace_run_or_join",True,1
7,E07::qwen_rag_top5,train::94::nda-16,NotMentioned,Entailment,False->False,"line_break_or_tab,whitespace_run_or_join",True,1
8,E07::qwen_rag_top5,train::350::nda-10,NotMentioned,Contradiction,False->False,"line_break_or_tab,whitespace_run_or_join",True,1
9,E08B::gpt5mini_rag_top5,train::352::nda-12,Entailment,Entailment,False->True,"line_break_or_tab,whitespace_run_or_join",True,1


## 7. Decision-stability checks (frozen rules re-applied offline; NOT new benchmarks)
### E13 (FULL vs RAG)

In [6]:
print(json.dumps(ST['E13'],indent=1))

{
 "v1": {
  "net_joint": -3,
  "guards_fail": {
   "contradiction_ge3_below": false,
   "macro_f1_below_0.03": false,
   "evidence_recall_below_3pp": true,
   "evidence_precision_below_3pp": true
  },
  "outcome": "B",
  "evidence_recall_delta_pp": -6.000000000000005,
  "evidence_precision_delta_pp": -4.160887656033285
 },
 "v2": {
  "net_joint": -6,
  "guards_fail": {
   "contradiction_ge3_below": false,
   "macro_f1_below_0.03": false,
   "evidence_recall_below_3pp": true,
   "evidence_precision_below_3pp": true
  },
  "outcome": "B",
  "evidence_recall_delta_pp": -9.000000000000007,
  "evidence_precision_delta_pp": -7.018030513176143
 },
 "stored_outcome": "B - FULL MATERIALLY BETTER",
 "v1_reproduces_stored_outcome": true,
 "decision_changes_under_v2": false,
 "FULL": {
  "joint_v1": 116,
  "joint_v2": 120,
  "evidence_recall_v1": 0.9,
  "evidence_recall_v2": 0.93,
  "evidence_precision_v1": 0.8571428571428571,
  "evidence_precision_v2": 0.8857142857142857
 },
 "RAG": {
  "joint_v

### E12B prompt statuses and E12C confirmation

In [7]:
print(json.dumps(ST['E12B'],indent=1)); print(json.dumps(ST['E12C'],indent=1))

{
 "gpt_p1": {
  "stored_status": "NO BENEFIT",
  "v1_net": -2,
  "v2_net": -3,
  "v1_status_recomputed": "NO BENEFIT",
  "v2_status": "NO BENEFIT",
  "evidence_recall_delta_pp_v1_v2": [
   0.0,
   -1.0000000000000009
  ],
  "evidence_precision_delta_pp_v1_v2": [
   -2.3355925993771676,
   -3.3339439457776066
  ]
 },
 "gpt_p2": {
  "stored_status": "HARMFUL",
  "v1_net": -8,
  "v2_net": -9,
  "v1_status_recomputed": "HARMFUL",
  "v2_status": "HARMFUL",
  "evidence_recall_delta_pp_v1_v2": [
   0.0,
   -1.0000000000000009
  ],
  "evidence_precision_delta_pp_v1_v2": [
   -3.820568140956482,
   -4.836389787846096
  ]
 },
 "gpt_p3": {
  "stored_status": "NEAR-TIE",
  "v1_net": 8,
  "v2_net": 6,
  "v1_status_recomputed": "NEAR-TIE",
  "v2_status": "NEAR-TIE",
  "evidence_recall_delta_pp_v1_v2": [
   7.000000000000006,
   5.000000000000004
  ],
  "evidence_precision_delta_pp_v1_v2": [
   5.937266616878267,
   3.995519044062734
  ]
 }
}
{
 "v1": {
  "net": -3,
  "er": -0.020000000000000018,
  

### E11 agent (A2 vs A3)

In [8]:
print(json.dumps(ST['E11'],indent=1))

{
 "v1": {
  "a2_success": 111,
  "a3_success": 111,
  "net_joint": 0,
  "joint_fail_to_success": 1,
  "joint_success_to_fail": 1,
  "mcnemar_p": 1.0
 },
 "v2": {
  "a2_success": 113,
  "a3_success": 113,
  "net_joint": 0,
  "joint_fail_to_success": 1,
  "joint_success_to_fail": 1,
  "mcnemar_p": 1.0
 },
 "e11_decision_C_rests_on": "net joint exactly zero (one recovery cancelled by one regression), no tools exercised"
}


## 8. Retrieval metrics
Recall@K/MRR use chunk-vs-gold character-offset overlap (`evaluation.scorer.map_chunks_to_gold_span_indices`), never text matching → unaffected; unchanged.
## 9. Joint semantics / threshold
Unchanged: E/C = correct label + gold-span recall ≥ τ=0.5 (empty gold → label suffices); NotMentioned = correct label + no predicted spans. τ not tuned.
## 10. Residual limitations (outside frozen v2)
Curly-vs-straight quotes, ellipsis-joined quotes, paraphrase/truncation and PDF artefacts still fail to map; long merged quotes covering many spans are not penalised (pre-existing).
## 11. Final evaluator status
**A — EVIDENCE_EVALUATOR_V2 APPROVED** (all six acceptance criteria met; architecture-neutral; no frozen decision changes).